# Extracting Structured Data with Gaffa's parse_json Action

In this notebook, we'll use Gaffa's `parse_json` action to extract structured data from Wikipedia's Python programming language page — no CSS selectors, no HTML parsing. We simply describe what data we want and the Gaffa returns it as a clean JSON object.

## Step 1: Install Dependencies

In [ ]:
!pip install requests

## Step 2: Set Up API Keys

## *If you are using Google Colab: Run the cell below, then skip to Step 3.*

In [ ]:
import os
import requests
import json
from google.colab import userdata

GAFFA_API_KEY = userdata.get("GAFFA_API_KEY")

if not GAFFA_API_KEY:
    raise RuntimeError("GAFFA_API_KEY is not set.")

print("✅ API key loaded successfully.")

✅ API key loaded successfully.


## *If you are using an IDE and already have a `.env` file: Run the next two cells, then continue to Step 3.*

In [ ]:
# !pip install python-dotenv

In [ ]:
# import os
# import requests
# import json
# from dotenv import load_dotenv

# load_dotenv()

# GAFFA_API_KEY = os.getenv("GAFFA_API_KEY")

# if not GAFFA_API_KEY:
#     raise RuntimeError("GAFFA_API_KEY is not set.")

# print("✅ API key loaded successfully.")

## Step 3: Define the Schema and Send the Request

We define a schema that tells Gaffa exactly what data we want to extract from the page. No selectors needed — we just describe the fields we want and the AI figures out where they are on the page.

In [ ]:
def extract_with_parse_json(url):
    payload = {
        "url": url,
        "proxy_location": "us",
        "async": False,
        "max_cache_age": 0,
        "settings": {
            "record_request": False,
            "actions": [
                {
                    "type": "parse_json",
                    "data_schema": {
                        "name": "WikipediaPythonPage",
                        "description": "Extract key information about the Python programming language from its Wikipedia page.",
                        "fields": [
                            {
                                "type": "string",
                                "name": "title",
                                "description": "The title of the Wikipedia article."
                            },
                            {
                                "type": "string",
                                "name": "creator",
                                "description": "The person who created or designed Python."
                            },
                            {
                                "type": "string",
                                "name": "first_release_year",
                                "description": "The year Python was first released."
                            },
                            {
                                "type": "string",
                                "name": "summary",
                                "description": "A brief 2-3 sentence summary of what Python is."
                            },
                            {
                                "type": "array",
                                "name": "key_features",
                                "description": "A list of key features or characteristics of Python.",
                                "fields": [
                                    {
                                        "type": "string",
                                        "name": "feature",
                                        "description": "A single key feature of Python."
                                    }
                                ]
                            },
                            {
                                "type": "array",
                                "name": "popular_uses",
                                "description": "Common use cases or fields where Python is widely used.",
                                "fields": [
                                    {
                                        "type": "string",
                                        "name": "use_case",
                                        "description": "A single use case for Python."
                                    }
                                ]
                            }
                        ]
                    },
                    "model": "gpt-4o-mini",
                    "output_type": "inline"
                }
            ]
        }
    }

    headers = {
        "x-api-key": GAFFA_API_KEY,
        "Content-Type": "application/json"
    }

    print("Calling Gaffa API with parse_json action...")
    response = requests.post(
        "https://api.gaffa.dev/v1/browser/requests",
        json=payload,
        headers=headers
    )
    response.raise_for_status()

    return response.json()  # ✅ Now returns the full response

print("✅ extract_with_parse_json function defined.")

✅ extract_with_parse_json function defined.


## Call the function

In [ ]:
url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
full_response = extract_with_parse_json(url)

Calling Gaffa API with parse_json action...


## View the Full API Response


In [ ]:
# Full raw response
print("📦 Full API Response:\n")
print(json.dumps(full_response, indent=2))

📦 Full API Response:

{
  "data": {
    "id": "brq_VgiH9uDGswbzWpP2z6GZhMvqdoHGoC",
    "url": "https://en.wikipedia.org/wiki/Python_(programming_language)",
    "proxy_location": "us",
    "state": "completed",
    "credit_usage": 10,
    "http_status_code": 202,
    "from_cache": false,
    "started_at": "2026-03-10T10:18:27.2774843Z",
    "completed_at": "2026-03-10T10:18:55.9612394Z",
    "running_time": "00:00:28.6837551",
    "page_load_time": "00:00:02.8594566",
    "actions": [
      {
        "id": "act_VgiH9vPZGgHHs8cTLWtgPyAuHqRYit",
        "type": "parse_json",
        "timestamp": "2026-03-10T10:18:55.9612283Z",
        "output": {
          "title": "Python (programming language)",
          "creator": "Guido van Rossum",
          "first_release_year": "1991",
          "summary": "Python is a high-level, general-purpose programming language that emphasizes code readability and supports multiple programming paradigms, including procedural, object-oriented, and functiona

# Just the extracted data

In [ ]:
# Just the extracted data
print("\n✅ Extracted Data:\n")
result = full_response["data"]["actions"][0]["output"]
print(json.dumps(result, indent=2))


✅ Extracted Data:

{
  "title": "Python (programming language)",
  "creator": "Guido van Rossum",
  "first_release_year": "1991",
  "summary": "Python is a high-level, general-purpose programming language that emphasizes code readability and supports multiple programming paradigms, including procedural, object-oriented, and functional programming. It is dynamically typed and garbage-collected, making it a popular choice for beginners and experienced developers alike.",
  "key_features": [
    {
      "feature": "Multi-paradigm programming language"
    },
    {
      "feature": "Dynamically typed"
    },
    {
      "feature": "Garbage-collected"
    },
    {
      "feature": "Significant indentation for code blocks"
    },
    {
      "feature": "Extensive standard library"
    }
  ],
  "popular_uses": [
    {
      "use_case": "Web development"
    },
    {
      "use_case": "Data analysis and visualization"
    },
    {
      "use_case": "Machine learning and artificial intelligenc

## Step 5: Access Individual Fields

Now that we have the structured data, we can access any field directly — no parsing, no cleanup needed.

In [ ]:
print(f"Title: {result['title']}")
print(f"Created by: {result['creator']}")
print(f"First released: {result['first_release_year']}")
print(f"\nSummary:\n{result['summary']}")

print("\nKey Features:")
for item in result['key_features']:
    print(f"  - {item['feature']}")

print("\nPopular Uses:")
for item in result['popular_uses']:
    print(f"  - {item['use_case']}")

Title: Python (programming language)
Created by: Guido van Rossum
First released: 1991

Summary:
Python is a high-level, general-purpose programming language that emphasizes code readability and supports multiple programming paradigms, including procedural, object-oriented, and functional programming. It is dynamically typed and garbage-collected, making it a popular choice for beginners and experienced developers alike.

Key Features:
  - Multi-paradigm programming language
  - Dynamically typed
  - Garbage-collected
  - Significant indentation for code blocks
  - Extensive standard library

Popular Uses:
  - Web development
  - Data analysis and visualization
  - Machine learning and artificial intelligence
  - Scripting and automation
  - Scientific computing


## Bonus: Extracting Structured Data from a PDF

`parse_json` doesn't just work on web pages — it also works on online PDFs. Here we'll extract structured data from an academic paper PDF using the same approach. We add a `download_file` action first to fetch the PDF, then `parse_json` to extract the data.

In [ ]:
def extract_from_pdf(url):
    payload = {
        "url": url,
        "proxy_location": None,
        "async": False,
        "max_cache_age": 0,
        "settings": {
            "record_request": False,
            "actions": [
                {
                    "type": "download_file"
                },
                {
                    "type": "parse_json",
                    "data_schema": {
                        "name": "AcademicPaper",
                        "description": "Schema for parsing academic paper summary and author information",
                        "fields": [
                            {
                                "type": "string",
                                "name": "title",
                                "description": "The full title of the academic paper"
                            },
                            {
                                "type": "string",
                                "name": "abstract",
                                "description": "The paper's abstract or summary"
                            },
                            {
                                "type": "array",
                                "name": "authors",
                                "description": "List of authors who contributed to the paper",
                                "fields": [
                                    {
                                        "type": "string",
                                        "name": "name",
                                        "description": "Author's full name as it appears in the paper"
                                    },
                                    {
                                        "type": "array",
                                        "name": "affiliations",
                                        "description": "Institutional affiliations for this author",
                                        "fields": [
                                            {
                                                "type": "string",
                                                "name": "institution",
                                                "description": "Name of the university or research institution"
                                            },
                                            {
                                                "type": "string",
                                                "name": "department",
                                                "description": "Department or division name"
                                            },
                                            {
                                                "type": "string",
                                                "name": "city",
                                                "description": "City where the institution is located"
                                            },
                                            {
                                                "type": "string",
                                                "name": "country",
                                                "description": "Country of the institution"
                                            }
                                        ]
                                    },
                                    {
                                        "type": "string",
                                        "name": "email",
                                        "description": "Author's contact email address if provided"
                                    }
                                ]
                            },
                            {
                                "type": "array",
                                "name": "keywords",
                                "description": "Key terms and topics covered in the paper",
                                "fields": [
                                    {
                                        "type": "string",
                                        "name": "keyword",
                                        "description": "Individual keyword or phrase"
                                    }
                                ]
                            }
                        ]
                    },
                    "instruction": "Parse this academic paper focusing on the title, abstract, author names, their institutional affiliations with department and location details, and their contact information.",
                    "model": "gpt-4o-mini",
                    "output_type": "inline",
                    "max_pages": 1
                }
            ]
        }
    }

    headers = {
        "x-api-key": GAFFA_API_KEY,
        "Content-Type": "application/json"
    }

    print("Calling Gaffa API to extract data from PDF...")
    response = requests.post(
        "https://api.gaffa.dev/v1/browser/requests",
        json=payload,
        headers=headers
    )
    response.raise_for_status()

    output = response.json()["data"]["actions"][1]["output"]
    return output

print("✅ extract_from_pdf function defined.")

✅ extract_from_pdf function defined.


## Run the PDF Extraction

Let's extract structured data from the academic paper PDF.

In [ ]:
pdf_url = "https://demo.gaffa.dev/simulate/pdf/ReasoningAboutActionAndChange.pdf"
pdf_result = extract_from_pdf(pdf_url)

print("✅ PDF data extracted successfully.\n")
print(json.dumps(pdf_result, indent=2))

Calling Gaffa API to extract data from PDF...
✅ PDF data extracted successfully.

{
  "title": "Reasoning about Action and Change",
  "abstract": "This chapter presents the state of research concerning the formalisation of an agent reasoning about a dynamic system which can be partially observed and acted upon. We first define the basic concepts of the area: system states, ontic and epistemic actions, observations; then the basic reasoning processes: prediction, progression, regression, postdiction, filtering, abduction, and extrapolation. We then recall the classical action representation problems and show how these problems are solved in some standard frameworks. For space reasons, we focus on these major settings: the situation calculus, STRIPS and some propositional action languages, dynamic logic, and dynamic Bayesian networks. We finally address a special case of progression, namely belief update.",
  "authors": [
    {
      "name": "Florence Dupin de Saint-Cyr",
      "affiliat

## Access Individual Fields from the PDF

Just like with web pages, we can access any field from the extracted PDF data directly.

In [ ]:
print(f"Title: {pdf_result['title']}")
print(f"\nAbstract:\n{pdf_result['abstract']}")

print("\nAuthors:")
for author in pdf_result['authors']:
    print(f"\n  Name: {author['name']}")
    if author.get('email'):
        print(f"  Email: {author['email']}")
    for affiliation in author.get('affiliations', []):
        print(f"  Institution: {affiliation.get('institution', 'N/A')}")
        print(f"  Department: {affiliation.get('department', 'N/A')}")
        print(f"  Location: {affiliation.get('city', 'N/A')}, {affiliation.get('country', 'N/A')}")

print("\nKeywords:")
for item in pdf_result.get('keywords', []):
    print(f"  - {item['keyword']}")

Title: Reasoning about Action and Change

Abstract:
This chapter presents the state of research concerning the formalisation of an agent reasoning about a dynamic system which can be partially observed and acted upon. We first define the basic concepts of the area: system states, ontic and epistemic actions, observations; then the basic reasoning processes: prediction, progression, regression, postdiction, filtering, abduction, and extrapolation. We then recall the classical action representation problems and show how these problems are solved in some standard frameworks. For space reasons, we focus on these major settings: the situation calculus, STRIPS and some propositional action languages, dynamic logic, and dynamic Bayesian networks. We finally address a special case of progression, namely belief update.

Authors:

  Name: Florence Dupin de Saint-Cyr
  Institution: IRIT-CNRS. Université Paul Sabatier
  Department: 
  Location: Toulouse, France

  Name: Andreas Herzig
  Institutio